In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer 
import numpy as np
import pandas as pd


In [ ]:

toy_corpus= ["the fat cat sat on the mat",
             "the big cat slept",
             "the dog chased a cat"]
vectorizer=TfidfVectorizer(use_idf=True)

corpus_tfidf=vectorizer.fit_transform(toy_corpus)

print(f"The vocabulary size is {len(vectorizer.vocabulary_.keys())} ")
print(f"The document-term matrix shape is {corpus_tfidf.shape}")

df=pd.DataFrame(np.round(corpus_tfidf.toarray(),2))
df.columns=vectorizer.get_feature_names_out()
df

In [ ]:
import requests

url  = "https://klassika.ru/proza/chehov/kashtanka.txt"
resp = requests.get(url)

In [ ]:
import re
import nltk

nltk.download('stopwords')
nltk.download('gutenberg')
nltk.download('punkt')
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [ ]:
nltk.download('punkt_tab')

In [ ]:
kashtanka = resp.text
kashtanka = kashtanka.split("\n")

In [ ]:

def preprocess_sentence(text):
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'[^\w\s]', '', text).lower()
    tokens = word_tokenize(text)
    # stop_words = set(stopwords.words('russian'))
    # filtered_tokens = [word for word in tokens if word not in stop_words]
    return tokens
kashtanka_words = [preprocess_sentence(text) for text in kashtanka]

In [ ]:
import nltk
from nltk.corpus import gutenberg
from nltk.lm import MLE
from nltk.lm.preprocessing import padded_everygram_pipeline


In [ ]:
model, vocab = padded_everygram_pipeline(2, kashtanka_words)

lm=MLE(2)
lm.fit(model,vocab)
print(list(lm.vocab)[:10])
print(f"The number of words is {len(lm.vocab)}")

In [ ]:
print(lm.generate(15, random_seed=42))

In [ ]:
lm.generate(100, text_seed=['<s>'], random_seed=42)

In [ ]:

macbeth = gutenberg.sents('shakespeare-macbeth.txt')

model, vocab = padded_everygram_pipeline(2, macbeth)
lm=MLE(2)
lm.fit(model,vocab)
print(list(lm.vocab)[:10])
print(f"The number of words is {len(lm.vocab)}")


In [ ]:
print(lm.generate(7, random_seed=42))

In [ ]:
import gensim

In [ ]:
from gensim.models import Word2Vec
model = Word2Vec(sentences=kashtanka_words, vector_size=100, window=4, min_count=2, workers=4, epochs=10)

In [ ]:
model.wv

In [ ]:
model.wv.similar_by_word('увидела',10)

In [ ]:
model.wv.key_to_index.keys()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import random
import numpy as np

np.random.seed(42)
words=list([e for e in model.wv.key_to_index.keys() if len(e)>3]) # plot words longer than 4
random.shuffle(words)
words3d = PCA(n_components=3, random_state=42).fit_transform(model.wv[words[:100]])

def plotWords3D(vecs, words, title):
    """
        Parameters
        ----------
        vecs : numpy-array
            Transformed 3D array either by PCA or other techniques
        words: a list of word
            the word list to be mapped
        title: str
            The title of plot     
        """
    fig = plt.figure(figsize=(14,10))
    ax = fig.add_subplot(111, projection='3d')
    for w, vec in zip(words, vecs):
        ax.text(vec[0],vec[1],vec[2], w, color=np.random.rand(3,))
    ax.set_xlim(min(vecs[:,0]), max(vecs[:,0]))
    ax.set_ylim(min(vecs[:,1]), max(vecs[:,1]))
    ax.set_zlim(min(vecs[:,2]), max(vecs[:,2]))
    ax.set_xlabel('DIM-1')
    ax.set_ylabel('DIM-2')
    ax.set_zlabel('DIM-3')
    plt.title(title)
    plt.show()
plotWords3D(words3d, words, "Visualizing Word2Vec Word Embeddings using PCA")

In [ ]:
fig = plt.figure(figsize=(14,10))


In [ ]:
fig.gca(projection="3d")

In [ ]:
from gensim.models import FastText
model = FastText(sentences=kashtanka_words, vector_size=100, window=5, min_count=5,  workers=4, epochs=10,word_ngrams=1)


np.random.seed(42)
words=[w[0] for w in model.wv.similar_by_word("Macbeth",50)]
words3d = PCA(n_components=3, random_state=42).fit_transform(model.wv[words])
plotWords3D(words3d, words, "Visualizing FastText Word Embeddings reduced by PCA")

In [ ]:
import numpy as np
import tensorflow as tf

In [ ]:
!wget  
!unzip SST-2.zip

In [ ]:
import pandas as pd 
df=pd.read_csv('SST-2/train.tsv',sep="\t")
sentences=df.sentence
labels=df.label

In [ ]:
max_sen_len=max([len(s.split()) for s in sentences])
words = ["PAD"]+list(set([w for s in sentences for w in s.split()]))
word2idx= {w:i for i,w in enumerate(words)}
max_words=max(word2idx.values())+1
idx2word= {i:w for i,w in enumerate(words)}
# preparing training set
train=[list(map(lambda x:word2idx[x], s.split())) for s in sentences]